In [1]:
"""
LSTM Flood Forecasting Model (PyTorch)
Predicts streamflow 24 hours (1 day) in advance using historical weather and streamflow data.
Target: streamflow_cfs_target_24h

Downloads preprocessed data from the W&B artifact produced by missouri_preprocessing.py
"""
import torch
import polars as pl
import wandb
import numpy as np
import joblib
from SRC.helper_functions.preprocessing import processor

config = {
    "input_cols": [
        "latitude",
        "longitude",
        "streamflow_cfs_mean",
        "gage_height_ft_mean",
        "precipitation_mm",
        "temperature_c",
        "specific_humidity_kgkg",
    ], 
    "target": "streamflow_cfs_target_24h",
    "train_split": 0.8,
    "val_split": 0.9,
    "sites": ["06923250", "06936530"],
    "file_path": "flood-dataset-missouri",
    "file_name": "flood_model_missouri",
    "table": "wandb.flood_model_missouri",
    "lag_window": 7,
    "frequency": "daily"
}

pcr = processor(config)
pcr.pull_wandb()
print(pcr.df["site_id"].unique())
print(pcr.df.shape)
train_X, val_X, test_X, train_y, val_y, test_y = pcr.return_outputs()

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from C:\Users\sacha\_netrc.
wandb: Downloading large artifact 'flood-dataset-missouri:latest', 604.73MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.3 (2043.0MB/s)


shape: (2,)
Series: 'site_id' [str]
[
	"06936530"
	"06923250"
]
(11929, 51)


In [2]:
# Test daily frequency - should only have noon observations
config_daily = {
    **config,
    "frequency": "daily",
    "sites": ["06923250"],
    "start_date": "2020-01-01",
    "end_date": "2020-12-31",
}
pcr_daily = processor(config_daily)
pcr_daily.pull_wandb()
print("Daily hours:", pcr_daily.df["observation_hour"].dt.hour().unique().to_list())
print("Daily rows:", pcr_daily.df.shape)
print("Daily shift (should be 1):", 1 if config_daily.get("frequency") == "daily" else 24)

wandb: Downloading large artifact 'flood-dataset-missouri:latest', 604.73MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.2 (3876.5MB/s)


Daily hours: [12]
Daily rows: (364, 51)
Daily shift (should be 1): 1


In [3]:
# Test hourly frequency - should have all hours
config_hourly = {
    **config,
    "frequency": "hourly",
    "sites": ["06923250"],
    "start_date": "2020-01-01",
    "end_date": "2020-01-07",  # short range to keep it small
}
pcr_hourly = processor(config_hourly)
pcr_hourly.pull_wandb()
print("Hourly hours:", pcr_hourly.df["observation_hour"].dt.hour().unique().sort().to_list())
print("Hourly rows:", pcr_hourly.df.shape)
print("Hourly shift (should be 24):", 1 if config_hourly.get("frequency") == "daily" else 24)

wandb: Downloading large artifact 'flood-dataset-missouri:latest', 604.73MB. 1 files...
wandb:   1 of 1 files downloaded.  
Done. 00:00:00.2 (3515.9MB/s)


Hourly hours: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Hourly rows: (145, 51)
Hourly shift (should be 24): 24


In [12]:
# Check basic structure
print("=== Row counts ===")
print(f"Train X: {train_X.shape}")
print(f"Val X:   {val_X.shape}")
print(f"Test X:  {test_X.shape}")
print(f"Train y: {train_y.shape}")
print(f"Val y:   {val_y.shape}")
print(f"Test y:  {test_y.shape}")

# Check split boundaries are chronological
print("\n=== Split boundaries ===")
print(f"Train:  {train_X['observation_hour'].min()} → {train_X['observation_hour'].max()}")
print(f"Val:    {val_X['observation_hour'].min()} → {val_X['observation_hour'].max()}")
print(f"Test:   {test_X['observation_hour'].min()} → {test_X['observation_hour'].max()}")

# Check no nulls
print("\n=== Null check ===")
print(train_X.null_count())

# Check scaler looks sane
print("\n=== Feature scaler ===")
print("Mean:", pcr.feature_scaler.mean_)
print("Scale:", pcr.feature_scaler.scale_)

=== Row counts ===
Train X: (9238, 39)
Val X:   (1156, 39)
Test X:  (1155, 39)
Train y: (9238, 1)
Val y:   (1156, 1)
Test y:  (1155, 1)

=== Split boundaries ===
Train:  2007-11-03 12:00:00+00:00 → 2021-01-29 12:00:00+00:00
Val:    2021-01-30 12:00:00+00:00 → 2022-10-20 12:00:00+00:00
Test:   2022-10-21 12:00:00+00:00 → 2024-06-26 12:00:00+00:00

=== Null check ===
shape: (1, 39)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ site_id ┆ observatio ┆ streamflo ┆ gage_heig ┆ … ┆ specific_ ┆ specific_ ┆ specific_ ┆ specific_ │
│ ---     ┆ n_hour     ┆ w_cfs_mea ┆ ht_ft_mea ┆   ┆ humidity_ ┆ humidity_ ┆ humidity_ ┆ humidity_ │
│ u32     ┆ ---        ┆ n         ┆ n         ┆   ┆ kgkg3     ┆ kgkg4     ┆ kgkg5     ┆ kgkg6     │
│         ┆ u32        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│         ┆            ┆ u32       ┆ u32       ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32       │
╞═════════╪